## Event Hub producer
Synthetic order events sent to the shared Event Hub.

In [ ]:
import json
import random
import time
from azure.eventhub import EventHubProducerClient, EventData

login = "lena066636"
scope_name = f"{login}-scope"

connection_str = dbutils.secrets.get(scope=scope_name, key="eventhub-connection-string")
eventhub_name = dbutils.secrets.get(scope=scope_name, key="eventhub-name")


In [ ]:
producer = EventHubProducerClient.from_connection_string(
    conn_str=connection_str, eventhub_name=eventhub_name)


In [ ]:
def make_event(i):
    return {
        "order_id": i,
        "customer": f"cust_{i % 50}",
        "amount": round(random.uniform(5, 200), 2),
        "ts": time.time(),
    }


In [ ]:
with producer:
    batch = producer.create_batch()
    for i in range(200):
        event = make_event(i)
        try:
            batch.add(EventData(json.dumps(event)))
        except ValueError:
            producer.send_batch(batch)
            batch = producer.create_batch()
            batch.add(EventData(json.dumps(event)))
    producer.send_batch(batch)
